# Sentiment Analysis Project
Logistic Regression + Naive Bayes + TextBlob
Dataset: Twitter (English)

In [ ]:

import pandas as pd

train_path = '/mnt/data/training.1600000.processed.noemoticon para IA.csv'
test_path = '/mnt/data/testdata.manual.2009.06.14.csv'

train_df = pd.read_csv(train_path, encoding='latin-1', header=None)
test_df = pd.read_csv(test_path, encoding='latin-1', header=None)

train_df.columns = ['polarity','id','date','query','user','text']
test_df.columns = ['polarity','id','date','query','user','text']

train_df.head(), test_df.head()


In [ ]:

import re
import spacy
nlp = spacy.load('en_core_web_sm')

def preprocess(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop]
    return ' '.join(tokens)

train_df['clean'] = train_df['text'].apply(preprocess)
test_df['clean'] = test_df['text'].apply(preprocess)
train_df.head()


In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(ngram_range=(1,2), min_df=5)
X_train = vectorizer.fit_transform(train_df['clean'])
X_test = vectorizer.transform(test_df['clean'])

y_train = train_df['polarity']
y_test = test_df['polarity']


In [ ]:

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

nb = MultinomialNB()
nb.fit(X_train, y_train)
pred_nb = nb.predict(X_test)

lr = LogisticRegression(max_iter=200)
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)

print("Naive Bayes:")
print(classification_report(y_test, pred_nb))

print("Logistic Regression:")
print(classification_report(y_test, pred_lr))


In [ ]:

from textblob import TextBlob

def tb_sentiment(x):
    pol = TextBlob(x).sentiment.polarity
    if pol > 0: return 4
    if pol < 0: return 0
    return 2

test_df['tb_pred'] = test_df['text'].apply(tb_sentiment)

print(classification_report(test_df['polarity'], test_df['tb_pred']))


## Wordcloud

In [ ]:

from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(width=800, height=400).generate(' '.join(train_df['clean'][:50000]))
plt.figure(figsize=(10,5))
plt.imshow(wc)
plt.axis('off')
